In [1]:
!pip -q install scikit-learn gradio

In [2]:
import random
import gradio as gr

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
intents = {
    "greeting": {
        "patterns": [
            "hi", "hello", "hey", "good morning",
            "good evening", "how are you", "hello there",
            "hey there"
        ],
        "responses": [
            "Hello! Welcome to Customer Support. How can I help you?",
            "Hi there! How may I assist you today?"
        ]
    },

    "order_status": {
        "patterns": [
            "where is my order", "track my order",
            "order tracking", "when will my order arrive",
            "check my delivery status", "my order is delayed",
            "order status please", "I want to track my package"
        ],
        "responses": [
            "You can track your order using the tracking link sent to your registered email or phone number.",
            "Please check your order confirmation message for the latest delivery status."
        ]
    },

    "return": {
        "patterns": [
            "how can I return a product", "I want to return my order",
            "return policy", "how to return an item",
            "can I exchange my product", "product replacement",
            "I received a damaged product", "return my item"
        ],
        "responses": [
            "You can request a return within 7 days of delivery, subject to our return conditions.",
            "Sorry about the issue! Please contact our support team with your order details."
        ]
    },

    "refund": {
        "patterns": [
            "where is my refund", "refund status",
            "when will I get my money", "I want my money back",
            "refund not received", "how long does refund take",
            "my refund is pending", "check refund status"
        ],
        "responses": [
            "Refunds are generally processed within 5-7 business days after approval.",
            "Please contact our support team with your order ID to check your refund status."
        ]
    },

    "shipping": {
        "patterns": [
            "what are shipping charges", "delivery charges",
            "how long does delivery take", "shipping information",
            "do you offer free delivery", "delivery time",
            "when will my package arrive", "shipping policy"
        ],
        "responses": [
            "Standard delivery generally takes 3-5 business days.",
            "Shipping charges depend on your location and order value."
        ]
    },

    "payment": {
        "patterns": [
            "what payment methods are available", "payment options",
            "can I pay using UPI", "credit card payment",
            "debit card payment", "payment failed",
            "online payment problem", "is cash on delivery available"
        ],
        "responses": [
            "We support UPI, debit cards, credit cards and other available payment options.",
            "If your payment failed, please check with your bank before trying again."
        ]
    },

    "product_info": {
        "patterns": [
            "tell me about your products", "product details",
            "what products do you sell", "product availability",
            "is this product available", "show me your products",
            "I need product information", "product features"
        ],
        "responses": [
            "We offer a variety of products. Please visit our product catalogue for details.",
            "Could you please specify which product you are interested in?"
        ]
    },

    "contact": {
        "patterns": [
            "how can I contact support", "customer care number",
            "contact customer service", "support email",
            "I need to speak to an agent", "help desk",
            "customer support contact", "talk to customer care"
        ],
        "responses": [
            "You can contact our customer support team at support@example.com.",
            "Our support team is available Monday to Saturday, 9 AM to 6 PM."
        ]
    },

    "thanks": {
        "patterns": [
            "thank you", "thanks", "thank you so much",
            "that was helpful", "great help", "appreciate it"
        ],
        "responses": [
            "You're welcome! Happy to help.",
            "Glad I could help you!"
        ]
    },

    "goodbye": {
        "patterns": [
            "bye", "goodbye", "see you",
            "talk to you later", "have a nice day",
            "exit", "see you later"
        ],
        "responses": [
            "Goodbye! Have a wonderful day.",
            "Thank you for contacting customer support. See you again!"
        ]
    }
}

print("Customer service dataset created successfully!")
print("Total intents:", len(intents))

Customer service dataset created successfully!
Total intents: 10


In [4]:
X = []
y = []

for intent, details in intents.items():
    for pattern in details["patterns"]:
        X.append(pattern)
        y.append(intent)

print("Total training examples:", len(X))
print("Total intent categories:", len(set(y)))

Total training examples: 77
Total intent categories: 10


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        sublinear_tf=True
    )),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

print("Chatbot model trained successfully!")

Chatbot model trained successfully!


In [6]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", round(accuracy * 100, 2), "%")
print("\nClassification Report:\n")

print(classification_report(
    y_test,
    y_pred,
    zero_division=0
))

Model Accuracy: 75.0 %

Classification Report:

              precision    recall  f1-score   support

     contact       1.00      1.00      1.00         2
     goodbye       1.00      1.00      1.00         1
    greeting       0.50      1.00      0.67         1
order_status       0.50      0.50      0.50         2
     payment       1.00      1.00      1.00         2
product_info       1.00      1.00      1.00         2
      refund       0.33      1.00      0.50         1
      return       1.00      0.50      0.67         2
    shipping       1.00      0.50      0.67         2
      thanks       0.00      0.00      0.00         1

    accuracy                           0.75        16
   macro avg       0.73      0.75      0.70        16
weighted avg       0.80      0.75      0.74        16



In [12]:
def chatbot_response(message, history=None):

    if not message.strip():
        return "Please enter a message."

    # Predict customer intent
    probabilities = model.predict_proba([message])[0]

    best_index = probabilities.argmax()
    confidence = probabilities[best_index]

    predicted_intent = model.classes_[best_index]

    # Display prediction details for debugging
    print("Customer Question:", message)
    print("Predicted Intent:", predicted_intent)
    print("Confidence Score:", round(confidence, 3))

    # Lower threshold for our small demonstration dataset
    if confidence < 0.10:
        return (
            "Sorry, I couldn't understand your question. "
            "Please contact our customer support team."
        )

    # Generate chatbot response
    response = random.choice(
        intents[predicted_intent]["responses"]
    )

    return response

In [13]:
demo = gr.ChatInterface(
    fn=chatbot_response,
    title="🤖 Customer Service Chatbot",
    description="Ask questions about orders, refunds, payments and shipping."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://355241d8da3ab69685.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
